# Analysis

## Imports

In [48]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['figure.dpi'] = 200

## Load data

In [49]:
train1_path = '../datasets/train_part_1.parquet'
train2_path = '../datasets/train_part_2.parquet'
train3_path = '../datasets/train_part_3.parquet'
labels_path = '../datasets/train_labels.parquet'

train1_df = pd.read_parquet(train1_path)
train2_df = pd.read_parquet(train2_path)
train3_df = pd.read_parquet(train3_path)
labels_df = pd.read_parquet(labels_path)

## Preprocess

#### Проверим размеры датасетов:

In [50]:
print(f"train1_df: {train1_df.shape}")
print(f"train1_df: {train2_df.shape}")
print(f"train1_df: {train3_df.shape}")
print(f"labels_df: {labels_df.shape}")

train1_df: (28618594, 23)
train1_df: (28558397, 23)
train1_df: (28500849, 23)
labels_df: (87514, 3)


#### Проверим колонки датасетов и их типы данных:

In [56]:
print("train1_df:")
for col in train1_df.columns:
    print(f"- {col}: {train1_df[col].dtype}")

print("\nlabels_df:")
for col in labels_df.columns:
    print(f"- {col}: {labels_df[col].dtype}")

train1_df:
- customer_id: int64
- event_id: int64
- event_dttm: str
- event_type_nm: int32
- event_desc: int32
- channel_indicator_type: int32
- channel_indicator_sub_type: int32
- operaton_amt: float64
- currency_iso_cd: float64
- mcc_code: str
- pos_cd: float64
- accept_language: str
- browser_language: str
- timezone: float64
- session_id: float64
- operating_system_type: float64
- battery: str
- device_system_version: str
- screen_size: str
- developer_tools: str
- phone_voip_call_state: float64
- web_rdp_connection: float64
- compromised: str
- target: float64

labels_df:
- customer_id: int64
- event_id: int64
- target: int32


#### Добавляем в датасеты признаков разметку по классам:

In [52]:
train1_df = pd.merge(train1_df, labels_df, on=['customer_id', 'event_id'], how='left')
train2_df = pd.merge(train2_df, labels_df, on=['customer_id', 'event_id'], how='left')
train3_df = pd.merge(train3_df, labels_df, on=['customer_id', 'event_id'], how='left')

In [53]:
print(train1_df['target'])

0          NaN
1          NaN
2          NaN
3          NaN
4          NaN
            ..
28618589   NaN
28618590   NaN
28618591   NaN
28618592   NaN
28618593   NaN
Name: target, Length: 28618594, dtype: float64


#### Заменяем пропущенные значения ("чистые" операции ранее имели NaN-значение):

In [54]:
train1_df['target'] = train1_df['target'].fillna(-1)
print(train1_df['target'])

0          -1.000
1          -1.000
2          -1.000
3          -1.000
4          -1.000
            ...  
28618589   -1.000
28618590   -1.000
28618591   -1.000
28618592   -1.000
28618593   -1.000
Name: target, Length: 28618594, dtype: float64


## Research 

#### Посмотрим, какое значение для анализа имеет каждый из признаков:

**Идентификационные данные**

`customer_id` - ID клиента  
`event_id` - ID операции  
`session_id` - ID сессии

Зачем: Позволяют выявлять аномальное поведение конкретного клиента или множественные операции в одной сессии

**Временные метки**

`event_dttm` - дата и время операции

Зачем: Фрод часто происходит в нерабочее время (ночью, в праздники)

**Типы операций**

`event_type_nm` - тип операции (например, покупка, перевод, снятие)  
`event_desc` - описание операции

Зачем: Некоторые типы операций более рискованны (например, крупные переводы)

**Каналы и устройства, с которых производится транзакция**

`channel_indicator_type` - тип канала (мобильное приложение, веб, терминал)  
`channel_indicator_sub_type` - подтип канала  
`operating_system_type` - ОС устройства  
`device_system_version` - версия ОС  
`screen_size` - размер экрана  
`battery` - уровень заряда батареи  
`phone_voip_call_state` - состояние VoIP звонка  
`browser_language` - язык браузера  
`accept_language` - Accept-Language заголовок  
`timezone` - часовой пояс  

Зачем:
* Несоответствие языка/часового пояса местоположению клиента
* Эмуляторы имеют специфические размеры экрана, уровень батареи (часто 100% или -1)
* VoIP состояние может указывать на подмену номера

**Финансовые данные**

`operaton_amt` - сумма операции  
`currency_iso_cd` - валюта

Зачем: Необычно крупные суммы, несоответствие валюты региону

**Геолокационные и торговые**

`mcc_code` - Merchant Category Code (код категории торговца)  
`pos_cd` - код точки продажи

Зачем: Подозрительные категории (казино, криптобиржи), несоответствие обычным паттернам покупок

**Технические признаки безопасности**

`developer_tools` - включены ли инструменты разработчика  
`web_rdp_connection` - подключение через RDP  
`compromised` - метка, что устройство/сессия скомпрометированы

Зачем:
* developer_tools = 1: попытка отладки/взлома
* web_rdp_connection = 1: удаленный доступ (частый признак мошенников)
* compromised = 1: уже известное скомпрометированное устройство

#### Проанализируем target:

Колонка `target` имеет значения 1, 0 и -1. Нам известно, что -1 - это самый часто встречающийся класс = "чистые" операции; два других означают подозрительные операции.

В задании указано, что **целевой класс, "красный свет" - это непотвержденные операции.**  
Это означает, что когда банк зафиксировал подозрительную операцию, то связался с клиентом и задал вопрос "Это вы совершили операцию?" и клиент ответил "Нет, это не я".  
**Это и является фродом.**

Также имеется "желтый свет" - операции, подтвержденные клиентом.  
Это означает, что когда банк зафиксировал подозрительную операцию, то клиент в ответ на вопрос банка подтвердил, что он сам совершил эту операцию.  
Это не целевой класс, не является фродом. Но он также может содержать выбросы по суммам или другие нетипичные значения в признаках.

В задании не указано, какой из классов 1 и 0 является подтвержденной клиентом операцией (НЕ фрод), а какой - неподтвержденной (фрод). Выясним это по косвенным признакам:

In [65]:
for val in [0, 1]:
    subset = train1_df[train1_df['target'] == val]
    print(f"Класс {val} (n = {len(subset)})")
    
    rdp_pct = subset['web_rdp_connection'].mean() * 100
    print(f"RDP: {rdp_pct:.2f}%")
    
    if 'developer_tools' in subset.columns:
        dev_tools_numeric = pd.to_numeric(subset['developer_tools'], errors='coerce')
        dev_pct = dev_tools_numeric.mean() * 100
        print(f"Developer tools: {dev_pct:.2f}%")

    if 'compromised' in subset.columns:
        comp_numeric = pd.to_numeric(subset['compromised'], errors='coerce')
        comp_pct = comp_numeric.mean() * 100
        print(f"Compromised: {comp_pct:.2f}%")
    
    # 4. Суммы
    print(f"Средняя сумма: {subset['operaton_amt'].mean():.0f}")
    print(f"Медианная сумма: {subset['operaton_amt'].median():.0f}")
    print(f"Максимальная сумма: {subset['operaton_amt'].max():.0f}\n")

Класс 0 (n = 12082)
RDP: 6.56%
Developer tools: 17.03%
Compromised: 0.17%
Средняя сумма: 22885937
Медианная сумма: 1010000
Максимальная сумма: 12762880000

Класс 1 (n = 17384)
RDP: 1.82%
Developer tools: 15.79%
Compromised: 0.03%
Средняя сумма: 7042597
Медианная сумма: 362916
Максимальная сумма: 1150380064



По классу 0 видим:
* более редкий
* чаще используются инструменты разработчика, и гораздо чаще (в 4 раза) - подключение через удаленный доступ
* больше скомпоментированных устройств
* значительно больше средняя, медианная и максимальная суммы

Делаем вывод, что именно этот класс означает неподтвержденные операции = является целевым.  
**Целевой класс - 0.**

## Visualization

## Test results